# 예제 02. 프로젝트 노트북 템플릿
빅데이터프로그래밍 · 13주차

이 노트북을 **복사해서** 자기 프로젝트를 만드세요. 일곱 단계 순서를 그대로 유지합니다.

```
1. 데이터 확인 → 2. 전처리 → 3. Dataset → 4. 모델
→ 5. 학습 → 6. 평가 → 7. 시각화
```

각 단계마다 **무엇을 확인해야 하는지** 주석으로 적어 두었습니다.


In [ ]:
# ===== 공통 준비 =====
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "/ torch", torch.__version__)


---
# 1. 데이터 확인

**확인할 것**
- 데이터 개수 (학습 / 검증)
- 입력 하나의 shape과 dtype
- 정답의 형태 (클래스 번호인가 실수인가)
- 클래스 분포 (한쪽으로 치우쳐 있지 않은가)
- 눈으로 본 표본 몇 개


In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_raw = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_raw  = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

CLASSES = ["티셔츠", "바지", "풀오버", "드레스", "코트",
           "샌들", "셔츠", "운동화", "가방", "앵클부츠"]

print("학습:", len(train_raw), "/ 검증:", len(test_raw))
x0, y0 = train_raw[0]
print("입력 shape:", tuple(x0.shape), "dtype:", x0.dtype)
print("값 범위:", float(x0.min()), "~", float(x0.max()))
print("정답:", y0, "→", CLASSES[y0])


In [ ]:
# 클래스 분포
counts = torch.bincount(train_raw.targets)
print("클래스별 개수:", counts.tolist())
print("가장 적은 클래스 비율:", f"{counts.min().item()/counts.sum().item():.4f}")

plt.figure(figsize=(8, 2.6))
plt.bar(range(10), counts)
plt.xticks(range(10), CLASSES, rotation=45, ha="right", fontsize=8)
plt.title("class distribution"); plt.tight_layout(); plt.show()


In [ ]:
# 표본 눈으로 확인
fig, axes = plt.subplots(2, 8, figsize=(15, 4))
for ax, i in zip(axes.flatten(), range(16)):
    im, lb = train_raw[i]
    ax.imshow(im.squeeze(), cmap="gray"); ax.set_title(CLASSES[lb], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()


---
# 2. 전처리

**확인할 것**
- 정규화를 했는가 (사전학습 모델을 쓸 때는 ImageNet 통계)
- 증강은 **학습 데이터에만** 적용했는가
- 검증 데이터는 흔들지 않았는가


In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
])
eval_tf = transforms.ToTensor()          # 검증은 변형 없이

train_set = datasets.FashionMNIST("./data", train=True,  download=True, transform=train_tf)
test_set  = datasets.FashionMNIST("./data", train=False, download=True, transform=eval_tf)

print("학습 변형:", train_tf)
print("\n검증 변형:", eval_tf)


---
# 3. Dataset과 DataLoader

**확인할 것**
- batch 하나의 shape
- 학습은 `shuffle=True`, 검증은 `shuffle=False`
- batch 수가 적당한가


In [ ]:
BATCH = 128

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print("입력:", tuple(xb.shape))
print("정답:", tuple(yb.shape), yb.dtype)
print("batch 수:", len(train_loader), "/", len(test_loader))


---
# 4. 모델

**확인할 것**
- 출력 뉴런 수 = 클래스 수
- 더미 입력으로 출력 shape 확인
- 파라미터 수
- 손실 함수가 문제 종류에 맞는가 (분류 CrossEntropy / 회귀 MSE)


In [ ]:
N_CLASSES = 10

class ProjectCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*7*7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


model = ProjectCNN().to(device)
print(model)

with torch.no_grad():
    out = model(torch.randn(4, 1, 28, 28).to(device))
print("\n출력:", tuple(out.shape))
assert out.shape == (4, N_CLASSES)
print("파라미터:", f"{sum(p.numel() for p in model.parameters()):,}개")


---
# 5. 학습

**확인할 것**
- 매 epoch `model.train()` / 평가 시 `model.eval()`
- `opt.zero_grad()` 를 빠뜨리지 않았는가
- epoch마다 학습·검증 손실과 정확도를 기록
- 학습 시간도 기록


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 8

def evaluate(loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


history = []
start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = loss_fn(model(x), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    tr, va = evaluate(train_loader), evaluate(test_loader)
    history.append((*tr, *va))
    print(f"epoch {epoch:2d}  학습 loss {tr[0]:.4f} acc {tr[1]:.4f}  |  "
          f"검증 loss {va[0]:.4f} acc {va[1]:.4f}")

elapsed = time.time() - start
print(f"\n학습 시간 {elapsed:.1f}초")


---
# 6. 평가

**확인할 것**
- 최종 검증 정확도가 성공 기준을 넘었는가
- 학습-검증 차이 (과적합 정도)
- 검증 손실이 가장 낮았던 epoch
- 클래스별 정확도


In [ ]:
tr_a = [h[1] for h in history]; va_a = [h[3] for h in history]
va_l = [h[2] for h in history]

best_epoch = int(np.argmin(va_l)) + 1
print(f"최종 검증 정확도  : {va_a[-1]:.4f}")
print(f"최고 검증 정확도  : {max(va_a):.4f}")
print(f"학습-검증 차이    : {tr_a[-1]-va_a[-1]:.4f}")
print(f"검증 손실 최소 epoch: {best_epoch}")

TARGET = 0.88                                # 계획서의 성공 기준
print(f"\n성공 기준 {TARGET}: {'달성' if max(va_a) >= TARGET else '미달'}")


In [ ]:
# 클래스별 정확도
model.eval()
correct = torch.zeros(N_CLASSES); total = torch.zeros(N_CLASSES)
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(dim=1).cpu()
        for c in range(N_CLASSES):
            m = y == c
            total[c] += m.sum(); correct[c] += (p[m] == c).sum()

cls_df = pd.DataFrame({"클래스": CLASSES, "개수": total.int().tolist(),
                       "정확도": [round(v, 4) for v in (correct/total).tolist()]})
print(cls_df.to_string(index=False))
print(f"\n가장 어려운 클래스: {CLASSES[int((correct/total).argmin())]}")


---
# 7. 시각화

**넣을 것**
- 학습·검증 손실 곡선
- 학습·검증 정확도 곡선
- 예측 결과 (맞은 것 · 틀린 것)
- 혼동행렬


In [ ]:
xs = range(1, len(history)+1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(xs, [h[0] for h in history], label="학습")
ax[0].plot(xs, [h[2] for h in history], label="검증")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, tr_a, label="학습"); ax[1].plot(xs, va_a, label="검증")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


In [ ]:
# 틀린 사례
model.eval()
imgs, trues, preds, probs = [], [], [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        p = out.argmax(dim=1).cpu(); pr = torch.softmax(out, dim=1).cpu()
        w = p != y
        if w.any():
            imgs.append(x[w]); trues.append(y[w]); preds.append(p[w])
            probs.append(pr[w].max(dim=1).values)
        if sum(len(t) for t in trues) > 16:
            break

imgs = torch.cat(imgs); trues = torch.cat(trues)
preds = torch.cat(preds); probs = torch.cat(probs)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax_, i in zip(axes.flatten(), range(16)):
    ax_.imshow(imgs[i].squeeze(), cmap="gray")
    ax_.set_title(f"{CLASSES[preds[i]]} ({probs[i]:.2f})\n정답 {CLASSES[trues[i]]}",
                  fontsize=8, color="crimson")
    ax_.axis("off")
plt.suptitle("틀린 예측", y=1.02); plt.tight_layout(); plt.show()


In [ ]:
# 혼동행렬
model.eval()
cm = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.int32)
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(dim=1).cpu()
        for t, pp in zip(y, p):
            cm[t, pp] += 1

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(10), CLASSES, rotation=45, ha="right", fontsize=8)
plt.yticks(range(10), CLASSES, fontsize=8)
plt.xlabel("예측"); plt.ylabel("정답"); plt.title("confusion matrix")
plt.tight_layout(); plt.show()

off = cm.clone()
off.fill_diagonal_(0)
top = off.flatten().argsort(descending=True)[:5]
print("가장 많이 헷갈린 쌍:")
for f in top:
    t, p = divmod(f.item(), N_CLASSES)
    print(f"  {CLASSES[t]} → {CLASSES[p]} : {off[t, p].item()}회")


---
# 결과 요약 — 발표 자료에 그대로 옮길 내용


In [ ]:
summary = pd.DataFrame([
    {"항목": "문제",            "내용": "의류 이미지 10종류 분류"},
    {"항목": "데이터",          "내용": f"학습 {len(train_set):,}장 / 검증 {len(test_set):,}장"},
    {"항목": "모델",            "내용": "CNN (Conv 2층 + BatchNorm + Dropout)"},
    {"항목": "파라미터",        "내용": f"{sum(p.numel() for p in model.parameters()):,}개"},
    {"항목": "학습 조건",       "내용": f"epoch {EPOCHS} · batch {BATCH} · Adam lr 1e-3"},
    {"항목": "최종 검증 정확도", "내용": f"{va_a[-1]:.4f}"},
    {"항목": "최고 검증 정확도", "내용": f"{max(va_a):.4f} (epoch {int(np.argmax(va_a))+1})"},
    {"항목": "학습-검증 차이",   "내용": f"{tr_a[-1]-va_a[-1]:.4f}"},
    {"항목": "학습 시간",       "내용": f"{elapsed:.1f}초"},
    {"항목": "가장 어려운 클래스", "내용": CLASSES[int((correct/total).argmin())]},
])
print(summary.to_string(index=False))


In [ ]:
torch.save(model.state_dict(), "project_model.pt")
print("모델 저장 완료")


## 마지막 확인 — 제출 전 체크리스트

- [ ] 런타임을 초기화하고 처음부터 끝까지 실행했는가
- [ ] 데이터와 레이블을 설명했는가
- [ ] 모델 구조를 설명했는가
- [ ] 학습 결과를 그래프로 제시했는가
- [ ] 틀린 예측을 분석했는가
- [ ] 성공 기준과 결과를 비교했는가
